In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("08-deduplication")
dfs = register_views(spark)
emp = dfs["employees"]
sal = dfs["salary_history"]
pa = dfs["project_assignments"]
po = dfs["purchase_orders"]

# ── Section 1: distinct() removes exact duplicates across ALL columns ──────────
# hist_id 15 and 16 have identical business data but different hist_id values,
# so distinct() will NOT remove them — all columns must match.
print("sal before distinct:", sal.count())         # 33
print("sal after distinct:", sal.distinct().count())  # 32 (one dup removed)
# Note: distinct() compares ALL columns — hist_id differs between rows 15 and 16,
# so it WON'T deduplicate them. Must exclude the surrogate key to find business-key dups.

# ── Section 2: dropDuplicates on business-key subset ──────────────────────────
biz_cols = ["emp_id", "salary_before", "salary_after", "effective_date", "change_reason", "changed_by"]
before = sal.count()
after = sal.dropDuplicates(subset=biz_cols).count()
print(f"sal before dropDuplicates(biz_key): {before}, after: {after}, removed: {before - after}")
# Expected: removed=1 (the dup for emp 5 on 2022-04-01)

# ── Section 3: Detect exact duplicates — GROUP BY + HAVING count > 1 ──────────
print("\n--- salary_history duplicates (business key) ---")
sal.groupBy(biz_cols) \
   .agg(F.count("*").alias("cnt"), F.collect_list("hist_id").alias("hist_ids")) \
   .filter(F.col("cnt") > 1) \
   .show(truncate=False)
# Expected: shows emp 5, 2022-04-01 dup with both hist_ids

# Same for purchase_orders
print("\n--- purchase_orders duplicates ---")
po_biz = ["dept_id", "vendor", "item_category", "amount", "order_date"]
po.groupBy(po_biz) \
  .agg(F.count("*").alias("cnt"), F.collect_list("order_id").alias("order_ids")) \
  .filter(F.col("cnt") > 1) \
  .show()

# Same for project_assignments
print("\n--- project_assignments duplicates ---")
pa_biz = ["emp_id", "project_id", "role", "start_date", "end_date", "hours_billed"]
pa.groupBy(pa_biz) \
  .agg(F.count("*").alias("cnt"), F.collect_list("assignment_id").alias("ids")) \
  .filter(F.col("cnt") > 1) \
  .show()

# ── Section 4: ROW_NUMBER dedup — keep lowest surrogate key ───────────────────
w_dup = Window.partitionBy(*biz_cols).orderBy("hist_id")
deduped_sal = sal.withColumn("rn", F.row_number().over(w_dup)) \
                 .filter(F.col("rn") == 1) \
                 .drop("rn")
print(f"ROW_NUMBER dedup: {sal.count()} → {deduped_sal.count()} rows")

# ── Section 5: Soft duplicate detection — same entity, different keys ─────────
# emp 18 and emp 19: same last_name, dept_id, hire_date, job_title; different emp_id and salary
print("\n--- soft duplicates in employees (Clark, dept 2, 2017-08-14) ---")
soft_dup_cols = ["last_name", "dept_id", "hire_date", "job_title"]
emp.groupBy(soft_dup_cols) \
   .agg(F.count("*").alias("cnt"),
        F.collect_list("emp_id").alias("emp_ids"),
        F.collect_list("salary").alias("salaries")) \
   .filter(F.col("cnt") > 1) \
   .show(truncate=False)
# Expected: surfaces emp 18 and emp 19 (Clark, dept 2, 2017-08-14)

# ── Section 6: Dedup keeping most complete record (fewest NULLs) ──────────────
null_score_cols = ["salary_before", "change_reason", "changed_by"]
w_complete = Window.partitionBy(*biz_cols).orderBy(
    F.col("null_score").asc(), F.col("hist_id").asc()
)
print("\n--- most complete record per business key ---")
sal.withColumn("null_score",
    sum(F.when(F.col(c).isNull(), 1).otherwise(0) for c in null_score_cols)
).withColumn("rn", F.row_number().over(w_complete)) \
 .filter(F.col("rn") == 1) \
 .drop("rn", "null_score") \
 .show(5)

# ── Section 7: Dedup latest record per entity (SCD-style) ─────────────────────
# Get the most recent salary entry per employee
w_latest = Window.partitionBy("emp_id").orderBy(
    F.col("effective_date").desc(), F.col("hist_id").desc()
)
current_salary = sal.withColumn("rn", F.row_number().over(w_latest)) \
                    .filter(F.col("rn") == 1) \
                    .drop("rn")
print("\n--- latest salary per employee ---")
current_salary.select("emp_id", "salary_after", "effective_date").orderBy("emp_id").show(10)

# ── Section 8: Write deduped result to Parquet ────────────────────────────────
out = output_path("parquet/salary_history_deduped")
deduped_sal.write.mode("overwrite").parquet(out)
print(f"Deduped salary_history written to: {out}")

stop_and_wait(spark)